In [ ]:
import concurrent.futures
import requests
import pandas as pd
import time
import os

## Criar a URL da propriedade e seus reviews, fazer a requisição e verificar se deu certo

In [ ]:
def RequisicaoProp(id_prop):
    url = f"https://prod.apigee.hostelworld.com/legacy-msapi-service/property/{id_prop}"

    # Cabeçalho essencial para não ser bloqueado pela API
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }

     # Tenta fazer a requisição até 3 vezes se houver lentidão ou queda rápida
    for tentativa in range(3):
        try:
            with requests.get(url, headers=headers, timeout=15, stream=True) as req:
                if req.status_code == 200:
                    return req.json()
                else:
                    return None

        except (requests.exceptions.Timeout, requests.exceptions.ConnectionError):
            # Se deu timeout ou erro de rede, espera 2 segundos e tenta de novo
            print("Erro de Timeout ou Conexão")
            time.sleep(2)
            continue 
        except Exception:
            # Outros erros inesperados saem do laço
            print("Erro")
            break

    return None

In [ ]:
def RequisicaoReviews(id_prop, pagina):
    url = f"https://prod.apigee.hostelworld.com/legacy-hwapi-service/2.2/properties/{id_prop}/reviews/?sort=-date&allLanguages=true&page={pagina}"

    # Cabeçalho essencial para não ser bloqueado pela API
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }

     # Tenta fazer a requisição até 3 vezes se houver lentidão ou queda rápida
    for tentativa in range(3):
        try:
            with requests.get(url, headers=headers, timeout=15, stream=True) as req:
                if req.status_code == 200:
                    return req.json()
                else:
                    return None

        except (requests.exceptions.Timeout, requests.exceptions.ConnectionError):
            # Se deu timeout ou erro de rede, espera 2 segundos e tenta de novo
            print("Erro de Timeout ou Conexão")
            time.sleep(2)
            continue 
        except Exception:
            # Outros erros inesperados saem do laço
            print("Erro")
            break

    return None

## Obter as Informações e os Reviews de cada Propriedade

In [ ]:
def ObterInfoProp(id_prop):
    df_prop = pd.json_normalize(RequisicaoProp(id_prop))

    # obtendo imagens
    img_prop = []
    for item in df_prop["images"][0]:
        prefixo = item['prefix']
        sufixo = item['suffix']
        # juntando os dois para formar a URL completa
        url_img_completa = f"{prefixo}{sufixo}"
        img_prop.append(url_img_completa)
    
    df_prop.at[0, "images"] = img_prop

    # obtendo decrição
    df_prop.at[0,"description"] = " ".join(df_prop["description"][0].replace("\r", " ").replace("\n", " ").split())

    # Serviços oferecidos
    servicos = df_prop["facilities"][0]
    servicos_prop = []
    for categoria_servico in servicos:
        facilities = []
        for servico in categoria_servico["facilities"]:
            facilities.append(servico["name"])
        servicos_prop.append(f"{categoria_servico["name"]}: {', '.join(facilities)}")
    
    df_prop.at[0, "facilities"] = servicos_prop

    return df_prop[["name", "images", "description", "ratings.overall", "facilities"]]

In [ ]:
# função para definir o número de páginas de reviews de cada propriedade
# necessária para fazer o loop da função que obtém as reviews
def ObterNumPaginasReviews(id_prop):
    dados = RequisicaoReviews(id_prop, 1)
    if not dados:
        return 0
    df = pd.json_normalize(dados)
    if "pagination.numberOfPages" in df.columns:
        val = df["pagination.numberOfPages"][0]
        if pd.notna(val):
            return int(val)
    return 0

In [ ]:
def ObterReviews(id_prop):
    # determinar o número de páginas
    num_paginas = ObterNumPaginasReviews(id_prop)

    # Cria um dataframe vazio caso não haja páginas ou reviews
    if num_paginas == 0:
        return pd.DataFrame(columns=["notes"])

    # cria um dataframe para armazenar todas as reviews
    df_final_reviews = pd.DataFrame()
    
    # varre todas as urls de reviews para obtê-las
    for pagina in range(1, num_paginas + 1):
        dados_reviews = RequisicaoReviews(id_prop, pagina)
        if dados_reviews and "reviews" in dados_reviews and dados_reviews["reviews"]:
            df_reviews = pd.json_normalize(dados_reviews["reviews"])
            df_final_reviews = pd.concat([df_final_reviews, df_reviews], ignore_index=True)
    
    # Se o dataframe final estiver vazio ou não tiver a coluna 'notes', retorna um DF vazio seguro
    if df_final_reviews.empty or "notes" not in df_final_reviews.columns:
        return pd.DataFrame(columns=["notes"])
    
    return df_final_reviews[["notes"]]

#### Função para chamar todas as outras e Fazer a Extração Completa

In [ ]:
def ExtracaoCompleta(id_prop):
    df_prop = ObterInfoProp(id_prop)
    df_reviews = ObterReviews(id_prop)

    # para juntar os df's é necessário criar uma coluna id_prop em cada um
    df_prop["id_prop"] = id_prop
    df_reviews["id_prop"] = id_prop
    
    # usando a função merge para unir os df's
    df_extracao_completa = pd.merge(df_reviews, df_prop, how="left", on=["id_prop"])
    
    return df_extracao_completa

# Paralelizar a extração usando a função

In [ ]:
# recommended is equal to the number of CPU Cores.
max_workers = 50

# número de extrações e tamanho de lote
num_extracoes = 1000
tamanho_lote = 100

# obter o tempo de início
time_init = time.time()

arquivo_csv = "dados_hostelworld.csv"

# limpa o arquivo velho toda vez que iniciar do zero
if os.path.exists(arquivo_csv):
    os.remove(arquivo_csv)

# paralelismo
for inicio in range(1, num_extracoes+1, tamanho_lote):
    # Define o fim do lote atual (garante que não ultrapasse o total máximo)
    fim = min(inicio + tamanho_lote - 1, num_extracoes)
    print(f"\n--- Processando Lote: IDs de {inicio} até {fim} ---")

    # cria uma lista de resultados para armazenar os df's resultantes do paralelismo
    lista_resultados = []
    
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        extracoes_futuras = {executor.submit(ExtracaoCompleta, id_prop): id_prop for id_prop in range(inicio, fim+1)}
        
        for i, extracao in enumerate(concurrent.futures.as_completed(extracoes_futuras)):
            try:
                resultado = extracao.result()
                if resultado is not None and not resultado.empty:
                    lista_resultados.append(resultado)
                else:
                    pass
            except Exception as e:
                pass
    
    # criando o DataFrame do lote
    if lista_resultados:
        df_lote = pd.concat(lista_resultados, ignore_index=True)

        # salvando df_lote em csv
        arquivo_existe = os.path.exists(arquivo_csv)
        df_lote.to_csv(arquivo_csv, mode='a', index=False, header=not arquivo_existe, encoding='utf-8-sig')
        print(f"Lote {inicio}-{fim} concluído e salvo com sucesso!")
    else:
        df_lote = pd.DataFrame()

# obter o tempo final e printar na tela
time_fim = time.time()
print(time_fim - time_init , ' segundos')